# Phase 1 - Self-Supervised Pre-Training

**Goal:** Train three SSL encoders on the PCam pretrain subset, save checkpoints.

**Checkpoints saved:**
- `checkpoints/simclr_encoder_tailored.pth` - SimCLR (tailored augmentations)
- `checkpoints/mae_encoder.pth` - MAE base (patch=8, depth=6)
- `checkpoints/mae_improved_encoder.pth` - MAE Improved (patch=4, depth=12, cosine LR, EMA)

**Runtime:** ~2–4 h per encoder on Colab T4. Run each section independently.

## 1 - Setup

Mount Google Drive, clone the GitHub repo, install dependencies.

In [ ]:
# Mount Google Drive (for PCam data access)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Clone the project repo
!git clone https://github.com/Friedrich-233/ST456_GroupProject.git /content/ST456_GroupProject

# Install missing dependency
!pip install timm -q

In [ ]:
import sys, os
sys.path.insert(0, '/content/ST456_GroupProject/code')

from pathlib import Path
import numpy as np

# Paths — edit this to your Drive folder location 
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/ST456 Group project/pcamv1')
DATA_DIR        = Path('/content/pcam_data')
CHECKPOINT_DIR  = Path('/content/ST456_GroupProject/checkpoints')
CHECKPOINT_DIR = Path('/content/ST456_GroupProject/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DRIVE_DATA_DIR : {DRIVE_DATA_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")

## 2 - Load Data

In [ ]:
from data import load_all_data, build_downstream_loaders, LABEL_FRACTIONS

print("Loading PCam data (this may take a few minutes)...")
data = load_all_data(
    drive_data_dir=DRIVE_DATA_DIR,
    data_dir=DATA_DIR,
    pretrain_fraction=0.15,        # ~50K images for SSL pre-training
    downstream_pool_fraction=0.15, # ~50K images for downstream fine-tuning pool
    use_subset=True,
)
print("\nData summary:")
data.summary()

In [ ]:
# Build downstream loaders 
train_loaders, val_loader, test_loader = build_downstream_loaders(
    data,
    label_fractions=LABEL_FRACTIONS,
)
print("Downstream loaders built successfully.")
for name, loader in train_loaders.items():
    print(f"  {name}: {len(loader.dataset)} samples")

## 3 - Pre-Train SimCLR (Tailored Augmentations)

H&E-specific augmentations: 90° rotation, H&E colour jitter, RandomResizedCrop.

In [ ]:
from training import train_simclr, SEED

# Tailored SimCLR
simclr_model, simclr_history, simclr_ckpt = train_simclr(
    data=data,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=10,
    batch_size=128,
    learning_rate=3e-4,
    tailored=True,
    checkpoint_name='simclr_encoder_tailored.pth',
)
print(f"\nSimCLR checkpoint saved → {simclr_ckpt}")

In [ ]:
# Plot SimCLR training loss
import pandas as pd, matplotlib.pyplot as plt

df = pd.DataFrame(simclr_history)
plt.figure(figsize=(8, 4))
plt.plot(df['epoch'], df['loss'], 'b-o')
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('SimCLR Training Curve')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

## 4 - Pre-Train MAE Base (patch=8, depth=6)

Reference MAE: smaller encoder, no warmup / EMA.

In [ ]:
from training import train_mae

mae_model, mae_history, mae_ckpt = train_mae(
    data=data,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=10,
    batch_size=128,
    learning_rate=1e-3,
    checkpoint_name='mae_encoder.pth',
)
print(f"\nMAE checkpoint saved -> {mae_ckpt}")

In [ ]:
# Plot MAE training loss
df = pd.DataFrame(mae_history)
plt.figure(figsize=(8, 4))
plt.plot(df['epoch'], df['loss'], 'r-o')
plt.xlabel('Epoch'); plt.ylabel('Reconstruction Loss')
plt.title('MAE (Base) Training Curve')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

## 5 - Pre-Train MAE Improved (patch=4, depth=12)

Key changes vs base:
- **patch_size 4** -> 576 patches per image (vs 144 for patch=8) captures cell-level details
- **depth 12** -> stronger encoder without fine-tune overfitting risk
- **Cosine LR + warmup** + **gradient clipping** -> stable training
- **EMA** -> smoother model weights

In [ ]:
from training import train_mae_improved

mae_imp_model, mae_imp_history, mae_imp_ckpt = train_mae_improved(
    data=data,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=15,
    batch_size=128,
    learning_rate=1e-3,
    warmup_epochs=2,
    weight_decay=0.05,
    mask_ratio=0.65,
    ema_decay=0.996,
    checkpoint_name='mae_improved_encoder.pth',
)
print(f"\nMAE Improved checkpoint saved -> {mae_imp_ckpt}")

In [ ]:
# Plot MAE Improved training loss
df = pd.DataFrame(mae_imp_history)
plt.figure(figsize=(8, 4))
ax = plt.subplot(111)
ax.plot(df['epoch'], df['loss'], 'g-o', label='MAE-Improved')
ax.set_xlabel('Epoch'); ax.set_ylabel('Reconstruction Loss')
ax.set_title('MAE Improved Training Curve')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6 - Checkpoint Summary

In [ ]:
import os

print("Pre-training Complete — Checkpoints")
for name, path in [
    ('SimCLR (tailored)',   CHECKPOINT_DIR / 'simclr_encoder_tailored.pth'),
    ('MAE (base)',          CHECKPOINT_DIR / 'mae_encoder.pth'),
    ('MAE Improved',        CHECKPOINT_DIR / 'mae_improved_encoder.pth'),
]:
    size_mb = os.path.getsize(path) / 1e6
    status = 'FOUND' if path.exists() else 'MISSING'
    print(f"  {status}  {name:<22}  {size_mb:.1f} MB  {path}")

print()
print("Next step: run 02_main_experiments.ipynb to fine-tune downstream classifiers.")